In [1]:
from google.colab import drive
import torch
import os
import glob
import numpy as np
import cv2

In [ ]:
#Mount Google Drive and navigate to the assignment directory
drive.mount('/content/drive')
%cd /content/drive/MyDrive/assignments/Video_action_recognition
!git pull
%pip install -q -r requirements.txt
%env PYTHONPATH=src
%env UCF50_N_FRAMES=16
with open("/content/drive/MyDrive/credentials/wandb_api.txt") as f:
    os.environ["WANDB_API_KEY"] = f.read().strip()
# Check for PyTorch and CUDA availability
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/assignments/Video_action_recognition
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 705 bytes | 9.00 KiB/s, done.
From https://github.com/ttl074/Video_action_recognition
   de6d607..2edb393  main       -> origin/main
Updating de6d607..2edb393
Fast-forward
 scripts/run.py |  2 +-
 src/models.py  | 16 +++++++++-------
 2 files changed, 10 insertions(+), 8 deletions(-)
env: PYTHONPATH=src
env: UCF50_N_FRAMES=16
PyTorch version: 2.11.0+cu130
CUDA available: True


In [3]:
# Setup paths for dataset
drive_path = '/content/drive/MyDrive/datasets/ucf50'
local_path = '/content'
url = 'https://www.crcv.ucf.edu/data/UCF50/UCF50.rar'
extract_path = os.path.join(local_path, 'datasets')
raw_dir = os.path.join(extract_path, 'UCF50')
frame_dir = os.path.join(extract_path,'UCF50_frames_16')

In [4]:
!python scripts/prep_dataset.py --drive_path {drive_path} --local_path {local_path} --url {url}

Local rar file already exists. Skipping download.
Extract path already exists at /content/datasets/UCF50. Skipping extraction.


In [5]:
video_counts = {}
frame_counts = []
failed_videos = []
for category in os.listdir('/content/datasets/UCF50'):
    category_path = os.path.join('/content/datasets/UCF50', category)
    video_count = 0
    for video in os.listdir(category_path):
        video_path = os.path.join(category_path, video)
        v_cap = cv2.VideoCapture(video_path)
        if not v_cap.isOpened():
            print("Failed to open video:", video_path)
            failed_videos.append(video_path)
            v_cap.release()
            continue
        v_len = int(v_cap.get(cv2.CAP_PROP_FRAME_COUNT))
        v_cap.release()
        frame_counts.append(v_len)
        video_count += 1
    video_counts[category] = video_count
largest_category = max(video_counts, key=video_counts.get)
smallest_category = min(video_counts, key=video_counts.get)
print("Number of videos:", sum(video_counts.values()))
print("Largest category:", largest_category, video_counts[largest_category])
print("Smallest category:", smallest_category, video_counts[smallest_category])
print("Median frame count:", np.median(frame_counts))
print("Min frame count:", np.min(frame_counts))
print("Max frame count:", np.max(frame_counts))
print("Failed videos:", failed_videos)

Number of videos: 6681
Largest category: HorseRiding 197
Smallest category: Skijet 100
Median frame count: 181.0
Min frame count: 29
Max frame count: 900
Failed videos: []


In [6]:
n_frames = 16
!python scripts/extract_frames.py --raw_dir {raw_dir} --n_frames {n_frames} --frame_dir {frame_dir}

Converting from avi: 100% 6681/6681 [00:00<00:00, 25295.81file/s]
Extracted 0 and skipped 6681 out of 6681 avis


In [13]:
!python -m pytest -v -rs tests/

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/assignments/Video_action_recognition
configfile: pytest.ini
plugins: typeguard-4.6.0, platformdirs-4.13.3, anyio-4.15.1, langsmith-0.14.11
collected 9 items                                                              

tests/test_dataset.py::test_raw_class_folders PASSED                     [ 11%]
tests/test_dataset.py::test_video_avis PASSED                            [ 22%]
tests/test_dataset.py::test_ucf50_frames_dir PASSED                      [ 33%]
tests/test_dataset.py::test_video_dirs PASSED                            [ 44%]
tests/test_dataset.py::test_frame_counts PASSED                          [ 55%]
tests/test_dataset.py::test_frames_read PASSED                           [ 66%]
tests/test_models.py::test_batch_independence PASSED                     [ 77%]
te

In [10]:
import sys, os
from collections import defaultdict
sys.path.insert(0, "src")          # %env PYTHONPATH doesn't affect the running kernel
import video_datasets as vd

frames_dir = "/content/datasets/UCF50_frames_16"
groups_per_class = defaultdict(set)
for cat in sorted(os.listdir(frames_dir)):
    for vid in os.listdir(os.path.join(frames_dir, cat)):
        groups_per_class[cat].add(vd.get_group(vid))

full = set(range(1, 26))
for cat, gs in groups_per_class.items():
    extra = sorted(gs - full)
    missing = sorted(full - gs)
    if extra or missing:
        print(f"{cat:22s} n={len(gs):2d} max={max(gs):2d} extra={extra} missing={missing}")

print("classes with exactly groups 1-25:",
      sum(gs == full for gs in groups_per_class.values()), "of", len(groups_per_class))

JumpRope               n=26 max=26 extra=[26] missing=[]
Nunchucks              n=28 max=28 extra=[26, 27, 28] missing=[]
PlayingTabla           n=28 max=28 extra=[26, 27, 28] missing=[]
PullUps                n=30 max=30 extra=[26, 27, 28, 29, 30] missing=[]
PushUps                n=26 max=26 extra=[26] missing=[]
RockClimbingIndoor     n=26 max=26 extra=[26] missing=[]
classes with exactly groups 1-25: 44 of 50


In [9]:
import os
from collections import Counter
frames_dir = "/content/datasets/UCF50_frames_16"

extra = Counter()
total = 0
for cat in sorted(os.listdir(frames_dir)):
    for vid in os.listdir(os.path.join(frames_dir, cat)):
        total += 1
        if vd.get_group(vid) > 25:
            extra[cat] += 1

print("total videos:", total)
print("videos in groups 26+:", sum(extra.values()))
print("total minus extras:", total - sum(extra.values()), "(reported: 6618)")
for cat, n in sorted(extra.items()):
    print(f"  {cat:22s} {n}")

total videos: 6681
videos in groups 26+: 63
total minus extras: 6618 (reported: 6618)
  JumpRope               4
  Nunchucks              18
  PlayingTabla           13
  PullUps                20
  PushUps                4
  RockClimbingIndoor     4
